# 🎬 News Video Pipeline — Colab GPU Worker

Mac 側オーケストレーター (`cli.ts --remote`) と **Google Drive フォルダ** を介して連携する GPU ワーカー。

- **入力** (Drive `news-video-pipeline/inbox/<jobId>/`): `script.txt`, `face.png`, `job.json`, (任意) `voice_ref.wav`
- **処理**: [1] Fishaudio で音声生成 → [2] LTX-2.3 (Lipdub LoRA) でリップシンク動画
- **出力** (Drive `news-video-pipeline/outbox/<jobId>/`): `narration.wav`, `talking.mp4`, `done.json`

**前提**: ランタイム = **A100**（メニュー → ランタイムのタイプを変更 → A100）。LTX-2.3 22B は 32GB+ VRAM 推奨。
Drive のマイドライブ直下に `news-video-pipeline` フォルダがある状態（Mac の rclone remote と同じ場所）。


In [ ]:
# === 1. Drive マウント & バス設定 =========================================
from google.colab import drive
drive.mount('/content/drive')

import os, pathlib
# Mac の rclone remote "gdrive:news-video-pipeline" と同じ場所を指す
BUS   = pathlib.Path('/content/drive/MyDrive/news-video-pipeline')
INBOX = BUS / 'inbox'
OUTBOX= BUS / 'outbox'
MODELS= BUS / 'models_cache'   # 重いモデルは Drive にキャッシュしてセッション間で再利用
for p in (INBOX, OUTBOX, MODELS): p.mkdir(parents=True, exist_ok=True)
print('BUS =', BUS)
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# === 2. ComfyUI + LTX-2.3 ノード ==========================================
%cd /content
![ -d ComfyUI ] || git clone https://github.com/comfyanonymous/ComfyUI
%cd /content/ComfyUI
!pip -q install -r requirements.txt
%cd /content/ComfyUI/custom_nodes
![ -d ComfyUI-LTXVideo ] || git clone https://github.com/Lightricks/ComfyUI-LTXVideo
%cd /content/ComfyUI/custom_nodes/ComfyUI-LTXVideo
!pip -q install -r requirements.txt
print('ComfyUI + LTXVideo ready')

In [ ]:
# === 3. LTX-2.3 モデル取得 (Drive キャッシュ → ComfyUI へ symlink) =========
# 公式リポジトリのファイル名は確認済み。HF repo id は配布元に合わせて調整可。
from huggingface_hub import hf_hub_download
import os, pathlib

# TODO: 配布元の HF repo id を確認して合わせる（Lightricks 公式 / Kijai/LTX2.3_comfy など）
LTX_REPO   = os.environ.get('LTX_REPO', 'Lightricks/LTX-2.3')
GEMMA_REPO = 'google/gemma-3-12b-it-qat-q4_0-unquantized'

CK = pathlib.Path('/content/ComfyUI/models')
FILES = {
    # filename : ComfyUI subfolder
    'ltx-2.3-22b-distilled-1.1.safetensors'        : 'checkpoints',
    'ltx-2.3-22b-ic-lora-lipdub-0.9.safetensors'   : 'loras',
    'ltx-2.3-spatial-upscaler-x2-1.1.safetensors'  : 'latent_upscale_models',
    'ltx-2.3-temporal-upscaler-x2-1.0.safetensors' : 'latent_upscale_models',
}
for fn, sub in FILES.items():
    dst_dir = CK / sub; dst_dir.mkdir(parents=True, exist_ok=True)
    cached = MODELS / fn
    if not cached.exists():
        print('downloading', fn)
        p = hf_hub_download(repo_id=LTX_REPO, filename=fn, local_dir=str(MODELS))
        cached = pathlib.Path(p)
    link = dst_dir / fn
    if not link.exists():
        os.symlink(cached, link)
print('LTX weights linked.')

# テキストエンコーダ (gemma) — フォルダごと
from huggingface_hub import snapshot_download
te_dir = CK / 'text_encoders' / 'gemma-3-12b-it-qat-q4_0-unquantized'
if not te_dir.exists():
    snapshot_download(repo_id=GEMMA_REPO, local_dir=str(te_dir))
print('text encoder ready:', te_dir)

In [ ]:
# === 4. Fishaudio (fish-speech) セットアップ ==============================
%cd /content
![ -d fish-speech ] || git clone https://github.com/fishaudio/fish-speech
%cd /content/fish-speech
!pip -q install -e .

# モデル取得（Drive キャッシュ）
from huggingface_hub import snapshot_download
import pathlib
FISH_CKPT = MODELS / 'fish-speech-1.5'
if not FISH_CKPT.exists():
    snapshot_download(repo_id='fishaudio/fish-speech-1.5', local_dir=str(FISH_CKPT))
print('fish-speech ckpt:', FISH_CKPT)

In [ ]:
# === 5. TTS 関数（Fishaudio）=============================================
# fish-speech は「semantic token 生成 → vocoder」の2段。バージョンで CLI 名が変わるので
# ここが最も調整が要る箇所。voice_ref があれば声質クローン、無ければデフォルト話者。
import subprocess, pathlib, shutil

def tts_fish(text: str, out_wav: str, voice_ref: str | None = None):
    work = pathlib.Path('/content/_tts'); work.mkdir(exist_ok=True)
    prompt_txt = work / 'prompt.txt'; prompt_txt.write_text(text, encoding='utf-8')
    ckpt = str(FISH_CKPT)
    # 1) text -> semantic tokens
    gen = ['python', '-m', 'tools.llama.generate',
           '--text', text, '--checkpoint-path', ckpt,
           '--num-samples', '1', '--output-dir', str(work)]
    if voice_ref:
        gen += ['--prompt-text', text, '--prompt-tokens', voice_ref]  # 参照音声を使う場合は要調整
    subprocess.run(gen, cwd='/content/fish-speech', check=True)
    # 2) tokens -> wav (vqgan / vocoder)
    subprocess.run(['python', '-m', 'tools.vqgan.inference',
                    '-i', str(work / 'codes_0.npy'),
                    '--checkpoint-path', ckpt,
                    '-o', out_wav],
                   cwd='/content/fish-speech', check=True)
    return out_wav

# 動作確認（任意）:
# tts_fish('こんばんは。AIニュースの時間です。', '/content/test.wav')

In [ ]:
# === 6. ComfyUI 起動 + API ヘルパ ========================================
import subprocess, time, requests, json, uuid, pathlib, urllib.parse

PORT = 8188
proc = subprocess.Popen(
    ['python', 'main.py', '--listen', '127.0.0.1', '--port', str(PORT)],
    cwd='/content/ComfyUI')
# 起動待ち
for _ in range(60):
    try:
        if requests.get(f'http://127.0.0.1:{PORT}/system_stats').ok: break
    except Exception: pass
    time.sleep(2)
print('ComfyUI up')

def queue_and_wait(workflow: dict, timeout=900):
    cid = uuid.uuid4().hex
    r = requests.post(f'http://127.0.0.1:{PORT}/prompt',
                      json={'prompt': workflow, 'client_id': cid})
    r.raise_for_status(); pid = r.json()['prompt_id']
    t0 = time.time()
    while time.time() - t0 < timeout:
        h = requests.get(f'http://127.0.0.1:{PORT}/history/{pid}').json()
        if pid in h: return h[pid]
        time.sleep(3)
    raise TimeoutError('ComfyUI render timed out')

In [ ]:
# === 7. リップシンク関数（LTX-2.3 Lipdub workflow）=======================
# 公式ノードに同梱のワークフロー JSON を API 形式で読み込み、画像/音声入力を差し替える。
import json, pathlib, shutil

WF_PATH = pathlib.Path('/content/ComfyUI/custom_nodes/ComfyUI-LTXVideo'
                       '/example_workflows/LTX-2.3_ICLoRA_Lipdub_Two_Stage_Distilled.json')
COMFY_INPUT = pathlib.Path('/content/ComfyUI/input'); COMFY_INPUT.mkdir(exist_ok=True)

def _patch_inputs(wf: dict, image_name: str, audio_name: str):
    """ワークフロー内の LoadImage / LoadAudio ノードに入力ファイル名を設定。
       ノード構造が違う場合は一度 ComfyUI 上で開いて class_type を確認して調整。"""
    nodes = wf.get('prompt', wf)  # API形式 or UI形式
    for nid, node in nodes.items():
        ct = node.get('class_type', '')
        ins = node.setdefault('inputs', {})
        if ct in ('LoadImage',):                 ins['image'] = image_name
        if ct in ('LoadAudio', 'VHS_LoadAudio'): ins['audio'] = audio_name
    return nodes

def lipsync_ltx(image_path: str, audio_path: str, out_mp4: str):
    img_name = 'face' + pathlib.Path(image_path).suffix
    aud_name = 'narration.wav'
    shutil.copy(image_path, COMFY_INPUT / img_name)
    shutil.copy(audio_path, COMFY_INPUT / aud_name)
    wf = json.loads(WF_PATH.read_text())
    prompt = _patch_inputs(wf, img_name, aud_name)
    hist = queue_and_wait(prompt)
    # 出力（SaveVideo / VHS_VideoCombine）のファイルを探して回収
    for nid, out in hist.get('outputs', {}).items():
        for key in ('gifs', 'videos', 'images'):
            for f in out.get(key, []):
                src = pathlib.Path('/content/ComfyUI/output') / f.get('subfolder','') / f['filename']
                if src.suffix.lower() in ('.mp4', '.webm', '.mov'):
                    shutil.copy(src, out_mp4); return out_mp4
    raise RuntimeError('LTX 出力動画が見つかりません（ワークフローの保存ノードを確認）')

In [ ]:
# === 8. ワーカーループ（Drive inbox を監視）==============================
import time, json, shutil, pathlib

def process_job(job_dir: pathlib.Path):
    spec = json.loads((job_dir / 'job.json').read_text())
    jid  = spec['jobId']
    print(f'▶ job {jid}')
    out = OUTBOX / jid; out.mkdir(parents=True, exist_ok=True)
    text = (job_dir / spec['script']).read_text(encoding='utf-8')
    face = job_dir / spec['face']
    ref  = job_dir / spec['voiceRef'] if spec.get('voiceRef') else None

    # [1] 音声生成
    wav = str(out / 'narration.wav')
    tts_fish(text, wav, str(ref) if ref else None)
    # [2] リップシンク
    lipsync_ltx(str(face), wav, str(out / 'talking.mp4'))
    # 完了マーカー（Mac はこれを見て pull する。必ず最後に書く）
    (out / 'done.json').write_text(json.dumps({'jobId': jid, 'ok': True}), encoding='utf-8')
    print(f'✓ done {jid}')

def worker_loop(poll=15):
    print('👀 inbox 監視開始 ...', INBOX)
    seen = set()
    while True:
        for job_dir in sorted(INBOX.glob('*')):
            jid = job_dir.name
            if jid in seen: continue
            if not (job_dir / 'job.json').exists(): continue
            if (OUTBOX / jid / 'done.json').exists():  # 既に処理済み
                seen.add(jid); continue
            try:
                process_job(job_dir); seen.add(jid)
            except Exception as e:
                print('✗ job failed:', jid, e)
                (OUTBOX / jid).mkdir(parents=True, exist_ok=True)
                (OUTBOX / jid / 'error.json').write_text(json.dumps({'error': str(e)}))
                seen.add(jid)
        time.sleep(poll)

# 実行（セルを止めるまで監視し続ける）:
worker_loop()